<a href="https://colab.research.google.com/github/ZulfiiaDitto/Evaluation-with-vLLM-/blob/main/medvlm_eval_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# medvlm-eval — self-contained Colab notebook


**Before you start**
1. Runtime → Change runtime type → **GPU** (prefer **L4 / 24 GB**; A100 only for 7B headroom).
2. Add an **`HF_TOKEN`** secret (key icon, left) and accept the Gemma + MedGemma licenses on the Hub.
3. Run top-to-bottom once.



In [24]:
%pip install -q "vllm==0.10.2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 155.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118

In [25]:
%pip install -q -U nvidia-nccl-cu13
# %pip install -q vllm \
#   --extra-index-url https://download.pytorch.org/whl/cu130 \
#   --extra-index-url https://pypi.nvidia.com

In [26]:
%pip install -q -U datasets

In [5]:
%pip install openai pillow pandas matplotlib pyyaml tqdm requests

## 1 · GPU check

In [6]:
import subprocess
# using nvidia cli command
out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    raise SystemExit("No GPU — Runtime > Change runtime type > GPU, then rerun.")
GPU_NAME = out.stdout.strip().splitlines()[0]
print("GPU:", GPU_NAME)

GPU: NVIDIA L4, 23034 MiB


## 3 · Hugging Face auth
Add **HF_TOKEN** in Colab Secrets first.

In [7]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_TOKEN"))
print("HF login ok")

HF login ok


## 4 · Mount Drive

In [8]:
# you may not need it if you using something different.
# but anyway your directories need to be customized

from google.colab import drive
drive.mount("/content/drive/")


Mounted at /content/drive/


In [9]:
%cd "/content/drive/MyDrive/Colab Notebooks/eval with vLLM"

/content/drive/MyDrive/Colab Notebooks/eval with vLLM


In [10]:
!ls

configs  figures  medvlm_eval_colab.ipynb  results  scripts  src


In [11]:
import os
DRIVE = "eval with vLLM"
for sub in ("results/accuracy","results/cost","results/robustness","figures"):
    os.makedirs(f"{sub}", exist_ok=True)


## 5 · Project
Make the dirs, `cd` in, and symlink `results/` + `figures/` to Drive so every script writes to persistent storage.

In [14]:
import os
PROJECT = "/content/drive/MyDrive/Colab Notebooks/eval with vLLM"
for sub in ("src","scripts","configs",
            "results/accuracy","results/cost","results/robustness","figures"):
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)
os.chdir(PROJECT)                  # absolute cd -> re-running can NEVER nest
os.environ["PROJECT"] = PROJECT    # so shell !cells can find it too
print("project (on Drive):", os.getcwd(), "| writable?", os.access(".", os.W_OK))

project (on Drive): /content/drive/MyDrive/Colab Notebooks/eval with vLLM | writable? True


## 6 · Project files
Each cell writes one file. Run them all once (top-to-bottom). To change behaviour later,
edit the cell and re-run it — that rewrites the file on disk.

**models.yaml** — the experimental matrix (reference; pairs isolate fine-tuning effect).

In [15]:
%%writefile configs/models.yaml
# this is just a config file -> you can add any model for comparison as you desire
# I am using the gemma and medgemma

defaults:
  port: 8000
  dtype: bfloat16
  max_model_len: 8192
  # one image per prompt is enough for VQA-RAD
  limit_mm_per_prompt: "image=1"

models:

  gemma3_4b:
    hf_id: google/gemma-3-4b-it
    family: general
    pair: gemma_medgemma          # cleanest isolation: same base architecture
    backend: vllm

  medgemma_4b:
    hf_id: google/medgemma-4b-it
    family: medical
    pair: gemma_medgemma
    backend: vllm
    # MedGemma applies CT windowing into RGB channels and 896x768 resize during
    # its own preprocessing. Note as a fairness caveat; do not silently normalize.

  qwen25vl_3b:
    hf_id: Qwen/Qwen2.5-VL-3B-Instruct
    family: general
    pair: anchor
    backend: vllm


Overwriting configs/models.yaml


**common.py** — shared engine: endpoint client, image encoding, fixed prompt, dataset loading, scoring. Everything imports this.

In [16]:
%%writefile src/common.py
"""Shared utilities for the medical-VLM evaluation pipeline.

Everything talks to a single OpenAI-compatible endpoint (served by vLLM),
so the accuracy leg and the cost leg measure the *same* server. That is the
property that makes the cost-vs-accuracy frontier a fair comparison.
"""
from __future__ import annotations

import base64
import io
import os
import re
from dataclasses import dataclass
from typing import Iterable

from openai import OpenAI

# vLLM exposes an OpenAI-compatible server.
#   vllm serve <model> --port 8000
# gives you http://localhost:8000/v1
# Cuz I am at the colab I am not able to access it


API_BASE = os.environ.get("OPENAI_API_BASE", "http://localhost:8000/v1")
API_KEY = os.environ.get("OPENAI_API_KEY", "EMPTY")  # per vLLM docs


def get_client() -> OpenAI:
    return OpenAI(base_url=API_BASE, api_key=API_KEY)

# Images

def pil_to_data_url(image, fmt: str = "PNG") -> str:
    """Encode a PIL image as a base64 data URL for the OpenAI vision format."""
    buf = io.BytesIO()
    if image.mode not in ("RGB", "L"):
        image = image.convert("RGB")
    image.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/{fmt.lower()};base64,{b64}"


# Querying a VLM
# Keep this prompt fixed across every model — prompt drift is the #1 cause of
# fake accuracy gaps.

CLOSED_INSTRUCTION = (
    "Answer the question about the medical image. "
    "Reply with a single word or short phrase only, no explanation."
)


def query_vlm(client: OpenAI, model: str, data_url: str, question: str,
              max_tokens: int = 32, temperature: float = 0.0) -> str:
    """One image + one question -> raw model text. temperature=0 for reproducibility."""
    resp = client.chat.completions.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": f"{CLOSED_INSTRUCTION}\n\nQuestion: {question}"},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
    )
    return (resp.choices[0].message.content or "").strip()


# Datasets
# utilizing the vqa-rad ds for evaluation of med images
# https://huggingface.co/datasets/flaviagiammarino/vqa-rad/viewer

# I added & commented Slake ds -> in case you want some variability

DATASET_IDS = {
    "vqa_rad": "flaviagiammarino/vqa-rad",
   # "slake":   "mdwiratathya/SLAKE-vqa-english",
}


@dataclass
class VQAExample:
    image: object
    question: str
    answer: str
    closed: bool # yes/no question


_YESNO = {"yes", "no"}


def load_medical_vqa(dataset: str, split: str = "test",
                     closed_only: bool = True) -> list[VQAExample]:
    """Load a medical VQA dataset and (optionally) keep the closed (yes/no) subset.

    Closed-set yes/no accuracy is exact-match and unambiguous — used as the
    primary metric.
    """
    from datasets import load_dataset

    if dataset not in DATASET_IDS:
        raise ValueError(f"Unknown dataset {dataset!r}; known: {list(DATASET_IDS)}")
    ds = load_dataset(DATASET_IDS[dataset], split=split)

    out: list[VQAExample] = []

    for row in ds:
        q = str(row.get("question", "")).strip()
        a = normalize_answer(str(row.get("answer", "")))
        img = row.get("image")
        if img is None or not q or not a:
            continue
        closed = a in _YESNO
        if closed_only and not closed:
            continue
        out.append(VQAExample(image=img, question=q, answer=a, closed=closed))
    return out

# Scoring

def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    # collapse leading yes/no so "yes, there is" scores as "yes"
    for tok in ("yes", "no"):
        if s == tok or s.startswith(tok + " "):
            return tok
    return s


def closed_match(pred: str, gold: str) -> bool:
    return normalize_answer(pred) == normalize_answer(gold)


Overwriting src/common.py


**derive_cost.py** — reduces the vllm-bench sweep to cost-per-1k-cases at the saturation rung.

In [17]:
%%writefile src/derive_cost.py
#!/usr/bin/env python
"""Reduce a vllm-bench-serve concurrency sweep to a single cost figure.

cost_per_1k_cases = (1000 / sustained_request_throughput) * (gpu_hourly / 3600)

We take the saturation point = the concurrency rung with the highest sustained
request throughput (req/s). That is the most favourable, defensible operating
point for each model and the right basis for a cost frontier.
"""
import argparse
import glob
import json
from pathlib import Path


def load_rungs(sweep_dir: str) -> list[dict]:
    rungs = []
    for path in sorted(glob.glob(str(Path(sweep_dir) / "*.json"))):
        try:
            d = json.loads(Path(path).read_text())
        except Exception:
            continue
        # vllm bench serve field names
        rps = d.get("request_throughput") or d.get("requests_per_second")
        if rps is None:
            continue
        rungs.append({
            "file": Path(path).name,
            "concurrency": d.get("max_concurrency") or d.get("num_concurrency"),
            "request_throughput": float(rps),
            "output_tok_throughput": d.get("output_throughput")
                                     or d.get("total_token_throughput"),
            "mean_ttft_ms": d.get("mean_ttft_ms"),
            "p99_ttft_ms": d.get("p99_ttft_ms"),
            "mean_e2e_ms": d.get("mean_e2el_ms") or d.get("mean_e2e_latency_ms"),
            "p99_e2e_ms": d.get("p99_e2el_ms") or d.get("p99_e2e_latency_ms"),
        })
    return rungs


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--sweep-dir", required=True)
    ap.add_argument("--gpu-hourly", type=float, required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    rungs = load_rungs(args.sweep_dir)
    if not rungs:
        raise SystemExit(f"No parseable bench results in {args.sweep_dir}")

    sat = max(rungs, key=lambda r: r["request_throughput"])
    rps = sat["request_throughput"]
    cost_per_1k = (1000.0 / rps) * (args.gpu_hourly / 3600.0)

    summary = {
        "model": args.model,
        "gpu_hourly_usd": args.gpu_hourly,
        "saturation_rung": sat,
        "sustained_request_throughput": round(rps, 3),
        "cost_per_1k_cases_usd": round(cost_per_1k, 4),
        "ttft_mean_ms": sat.get("mean_ttft_ms"),
        "ttft_p99_ms": sat.get("p99_ttft_ms"),
        "all_rungs": rungs,
    }
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(summary, indent=2))
    print(json.dumps({k: summary[k] for k in
                      ("model", "sustained_request_throughput",
                       "cost_per_1k_cases_usd")}, indent=2))


if __name__ == "__main__":
    main()


Overwriting src/derive_cost.py


**aggregate_results.py** — joins the three legs into `results/summary.csv`.

In [42]:
%%writefile src/aggregate_results.py
#!/usr/bin/env python
"""Combine the three evals into one tidy table: results/summary.csv.
"""
import json
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results"

# Family labels for plotting (general vs medical) — keep in sync with models.yaml.
# if you added more model in config.yml -> edit here as well
FAMILY = {
    "qwen25vl_3b": "general", "gemma3_4b": "general",
    "medgemma_4b": "medical",
}


def _load(dir_name: str) -> list[dict]:
    out = []
    for p in sorted((RES / dir_name).glob("*.json")):
        try:
            out.append(json.loads(p.read_text()))
        except Exception:
            pass
    return out


def main() -> None:
    acc = {(d["model"], d["dataset"]): d for d in _load("accuracy")}
    rob = {(d["model"], d["dataset"]): d for d in _load("robustness")}
    cost = {d["model"]: d for d in _load("cost")}

    rows = []
    for (model, dataset), a in acc.items():
        r = rob.get((model, dataset), {})
        c = cost.get(model, {})
        rows.append({
            "model": model,
            "family": FAMILY.get(model, "unknown"),
            "dataset": dataset,
            "accuracy": a.get("accuracy"),
            "n_acc": a.get("n"),
            "flip_rate": r.get("flip_rate"),
            "consistency": r.get("consistency"),
            "cost_per_1k_cases_usd": c.get("cost_per_1k_cases_usd"),
            "req_throughput": c.get("sustained_request_throughput"),

        })

    if not rows:
        raise SystemExit("No results found — run the accuracy leg first.")

    df = pd.DataFrame(rows).sort_values(["dataset", "accuracy"], ascending=[True, False])
    out = RES / "summary.csv"
    df.to_csv(out, index=False)
    print(df.to_string(index=False))
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main()


Overwriting src/aggregate_results.py


**run_accuracy.py** —  1: closed-set (yes/no) accuracy against the endpoint.

In [19]:
%%writefile scripts/run_accuracy.py
#!/usr/bin/env python
"""Accuracy -> closed-set (yes/no) medical VQA against the vLLM endpoint.
  python scripts/run_accuracy.py --model qwen25vl_7b --dataset vqa_rad
"""
import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import common

from tqdm import tqdm


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="served-model-name, e.g. qwen25vl_7b")
    ap.add_argument("--dataset", required=True, choices=list(common.DATASET_IDS))
    ap.add_argument("--split", default="test")
    ap.add_argument("--limit", type=int, default=0, help="cap examples (0 = all)")
    ap.add_argument("--outdir", default="results/accuracy")
    args = ap.parse_args()

    client = common.get_client()
    examples = common.load_medical_vqa(args.dataset, args.split, closed_only=True)
    if args.limit:
        examples = examples[: args.limit]
    if not examples:
        sys.exit("No closed-set examples loaded — check dataset id / split.")

    print(f">> {args.model} on {args.dataset}: {len(examples)} closed-set items")

    correct = 0
    rows = []
    t0 = time.time()
    for ex in tqdm(examples):
        data_url = common.pil_to_data_url(ex.image)
        try:
            pred = common.query_vlm(client, args.model, data_url, ex.question)
        except Exception as e:  # keep going, log the failure as a miss
            pred, err = "", str(e)
        else:
            err = None
        ok = common.closed_match(pred, ex.answer)
        correct += int(ok)
        rows.append({"q": ex.question, "gold": ex.answer,
                     "pred": pred, "correct": ok, "error": err})

    n = len(examples)
    summary = {
        "model": args.model,
        "dataset": args.dataset,
        "split": args.split,
        "n": n,
        "accuracy": round(correct / n, 4),
        "wall_seconds": round(time.time() - t0, 1),
        "metric": "closed_set_exact_match",
        "prompt": common.CLOSED_INSTRUCTION,
    }

    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    stem = f"{args.model}__{args.dataset}"
    (outdir / f"{stem}.json").write_text(json.dumps(summary, indent=2))
    with (outdir / f"{stem}.jsonl").open("w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


Overwriting scripts/run_accuracy.py


**paraphrase_robustness.py** — leg 3: paraphrase consistency / answer-flip rate.

In [20]:
%%writefile scripts/paraphrase_robustness.py
#!/usr/bin/env python
"""Robustness -> paraphrase consistency on closed-set questions.

Re-ask each yes/no question several ways and measure how often the answer flips.

  flip_rate   = fraction of questions where the model is NOT unanimous across
                the original + paraphrases (lower is better)
  consistency = 1 - flip_rate

  python scripts/paraphrase_robustness.py --model qwen25vl_7b --dataset vqa_rad --k 4
"""
import argparse
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import common

from openai import OpenAI
from tqdm import tqdm


def paraphrase_client() -> tuple[OpenAI, str]:
    base = os.environ.get("PARAPHRASE_API_BASE", common.API_BASE)
    model = os.environ.get("PARAPHRASE_MODEL", "")  # default set by caller
    return OpenAI(base_url=base, api_key=os.environ.get("OPENAI_API_KEY", "EMPTY")), model


def make_paraphrases(client: OpenAI, model: str, question: str, k: int) -> list[str]:
    """Ask an LLM for k meaning-preserving rephrasings; return [] on failure."""
    prompt = (
        f"Rewrite the following yes/no medical question in {k} different ways "
        f"that preserve its exact meaning. Return ONLY a JSON array of "
        f"{k} strings, no prose.\n\nQuestion: {question}"
    )
    try:
        resp = client.chat.completions.create(
            model=model, max_tokens=256, temperature=0.7,
            messages=[{"role": "user", "content": prompt}],
        )
        text = (resp.choices[0].message.content or "").strip()
        text = text[text.find("["): text.rfind("]") + 1]
        out = json.loads(text)
        return [str(s).strip() for s in out if str(s).strip()][:k]
    except Exception:
        return []


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="target VLM served-model-name")
    ap.add_argument("--dataset", required=True, choices=list(common.DATASET_IDS))
    ap.add_argument("--split", default="test")
    ap.add_argument("--k", type=int, default=4, help="paraphrases per question")
    ap.add_argument("--limit", type=int, default=200, help="0 = all")
    ap.add_argument("--outdir", default="results/robustness")
    args = ap.parse_args()

    vlm = common.get_client()
    pp_client, pp_model = paraphrase_client()
    if not pp_model:
        pp_model = args.model  # reuse the target model for paraphrasing

    examples = common.load_medical_vqa(args.dataset, args.split, closed_only=True)
    if args.limit:
        examples = examples[: args.limit]
    if not examples:
        sys.exit("No closed-set examples loaded.")

    print(f">> robustness: {args.model} on {args.dataset}, "
          f"{len(examples)} questions x (1+{args.k}) phrasings")

    flips = 0
    majority_correct = 0
    rows = []
    t0 = time.time()
    for ex in tqdm(examples):
        data_url = common.pil_to_data_url(ex.image)
        variants = [ex.question] + make_paraphrases(pp_client, pp_model, ex.question, args.k)
        answers = []
        for q in variants:
            try:
                pred = common.query_vlm(vlm, args.model, data_url, q)
            except Exception:
                pred = ""
            answers.append(common.normalize_answer(pred))
        unanimous = len(set(answers)) == 1
        flips += int(not unanimous)
        majority = Counter(answers).most_common(1)[0][0]
        majority_correct += int(common.closed_match(majority, ex.answer))
        rows.append({"q": ex.question, "gold": ex.answer,
                     "variants": variants, "answers": answers,
                     "unanimous": unanimous})

    n = len(examples)
    summary = {
        "model": args.model,
        "dataset": args.dataset,
        "k_paraphrases": args.k,
        "n": n,
        "flip_rate": round(flips / n, 4),
        "consistency": round(1 - flips / n, 4),
        "majority_vote_accuracy": round(majority_correct / n, 4),
        "wall_seconds": round(time.time() - t0, 1),
    }

    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    stem = f"{args.model}__{args.dataset}"
    (outdir / f"{stem}.json").write_text(json.dumps(summary, indent=2))
    with (outdir / f"{stem}.jsonl").open("w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


Overwriting scripts/paraphrase_robustness.py


**run_cost.sh** — leg 2: `vllm bench serve` concurrency sweep, then calls derive_cost.py.

In [21]:
%%writefile scripts/run_cost.sh
#!/usr/bin/env bash
# Cost -> sweep load with vLLM's built-in serving benchmark, against the
# SAME running server the accuracy used.
# see vLLM library docs for more details.
#   MODEL_HF="Qwen/Qwen2.5-VL-3B-Instruct" ./scripts/run_cost.sh qwen25vl_3b 0.80
set -euo pipefail

KEY="${1:-}"
GPU_HOURLY="${2:-2.50}"  # your provider's $/GPU-hour (L4 ~0.80, A100 ~3.00)
NUM_PROMPTS="${NUM_PROMPTS:-200}"
MODEL_HF="${MODEL_HF:-$KEY}"  # REAL model id/path for the tokenizer
PORT="${PORT:-8000}"
OUTDIR="results/cost/raw/${KEY}"

if [[ -z "$KEY" ]]; then echo "usage: $0 <model_key> [gpu_hourly_usd]" >&2; exit 1; fi
if [[ "$MODEL_HF" == "$KEY" ]]; then
  echo "WARN: MODEL_HF not set — using '$KEY' as tokenizer, which will 404." >&2
  echo "      Re-run with: MODEL_HF=\"<hf_id_or_local_path>\" $0 $KEY $GPU_HOURLY" >&2
fi
mkdir -p "$OUTDIR"

# Throughput is a curve: sweep concurrency until it plateaus; derive_cost picks
# the saturation rung.
for C in 1 8 32 64; do
  echo ">> $KEY @ concurrency=$C"
  vllm bench serve \
    --backend openai-chat \
    --base-url "http://localhost:${PORT}" \
    --endpoint /v1/chat/completions \
    --model "$KEY" \
    --tokenizer "$MODEL_HF" \
    --dataset-name random-mm \
    --num-prompts "$NUM_PROMPTS" \
    --max-concurrency "$C" \
    --save-result \
    --result-dir "$OUTDIR" \
    --result-filename "c${C}.json" \
  || echo "   (rung c=$C failed; continuing)"
done

# Reduce the sweep to one cost-per-1k-cases figure (saturation rung).
python src/derive_cost.py \
  --model "$KEY" \
  --sweep-dir "$OUTDIR" \
  --gpu-hourly "$GPU_HOURLY" \
  --out "results/cost/${KEY}.json"

Overwriting scripts/run_cost.sh


Make the shell script executable and verify the DATASET_IDS you'll rely on.

In [22]:
!chmod +x scripts/run_cost.sh
import sys; sys.path.insert(0, "src")
import common
print("Datasets to verify resolve on HF:")
for k, v in common.DATASET_IDS.items():
    print(f"  {k:10s} -> {v}")
print("\nPrompt used for every model:\n ", common.CLOSED_INSTRUCTION)

Datasets to verify resolve on HF:
  vqa_rad    -> flaviagiammarino/vqa-rad

Prompt used for every model:
  Answer the question about the medical image. Reply with a single word or short phrase only, no explanation.


## 7 · Stamp run metadata
Records GPU + vLLM version beside results — part of the reproducibility story.

In [27]:
import json, datetime, vllm
json.dump({"gpu": GPU_NAME, "vllm": vllm.__version__,
           "timestamp": str(datetime.datetime.now())},
          open(f"{PROJECT}/results/run_meta.json", "w"), indent=2)
!pip freeze > "$PROJECT/results/requirements.lock"
print("saved to", f"{PROJECT}/results/")

saved to /content/drive/MyDrive/Colab Notebooks/eval with vLLM/results/


## 8 · Server helper (background vLLM)

In [28]:
import subprocess, time, requests
_SERVER = None
LOG = "/content/vllm.log"

def serve(model_key, hf_id, max_len=4096, mem_util=0.90, port=8000, quantization=None):
    global _SERVER
    stop()
    subprocess.run(f"fuser -k {port}/tcp", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    logf = open(LOG, "w")
    args = ["vllm","serve",hf_id,"--served-model-name",model_key,"--port",str(port),
            "--max-model-len",str(max_len),"--gpu-memory-utilization",str(mem_util),
            "--limit-mm-per-prompt",'{"image": 1}']
    if quantization:
        args += ["--quantization", quantization]          # <-- new
    _SERVER = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT)
    print(f"starting {model_key} (quant={quantization}) ... (log -> {LOG})")
    for _ in range(180):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=3).status_code == 200:
                print("server up"); return
        except Exception: pass
        if _SERVER.poll() is not None:
            print(f"\n--- vLLM exited (code {_SERVER.returncode}); last 40 log lines ---")
            print("".join(open(LOG).readlines()[-40:]))
            raise RuntimeError(f"vllm exited; see {LOG}")
        time.sleep(5)
    raise RuntimeError("server did not become healthy in time")

def stop():
    global _SERVER
    if _SERVER and _SERVER.poll() is None:
        _SERVER.terminate()
        try: _SERVER.wait(timeout=30)
        except Exception: _SERVER.kill()
    _SERVER = None

## 9 · Per-model run  ⟳ repeat for each model
Set `MODEL_KEY` / `HF_ID`, run the four cells; the server stops at the end. Then change the
two values and run again.

Keys: `qwen25vl_3b`,  `gemma3_4b`, `medgemma_4b`

- **7B on L4:** keep `MAX_LEN=4096`. For headroom, run 7B in an A100 session.


In [ ]:
# full precision
# Running medgemma

MODEL_KEY, HF_ID = "medgemma_4b", "google/medgemma-4b-it"
#LOCAL = prepare_model(HF_ID)
serve("medgemma_4b", "google/medgemma-4b-it", max_len=2048, mem_util=0.95)

starting medgemma_4b (quant=None) ... (log -> /content/vllm.log)
server up


In [ ]:
# Accuracy — start with --limit 30
!python scripts/run_accuracy.py --model $MODEL_KEY --dataset vqa_rad --limit 30


README.md: 100% 3.91k/3.91k [00:00<00:00, 6.99MB/s]
data/train-00000-of-00001-eb8844602202be(…): 100% 24.2M/24.2M [00:01<00:00, 14.7MB/s]
data/test-00000-of-00001-e5bc3d208bb4dee(…): 100% 10.3M/10.3M [00:01<00:00, 7.74MB/s]
Generating train split: 100% 1793/1793 [00:00<00:00, 6732.95 examples/s]
Generating test split: 100% 451/451 [00:00<00:00, 8849.28 examples/s]
>> medgemma_4b on vqa_rad: 30 closed-set items
100% 30/30 [00:12<00:00,  2.49it/s]
{
  "model": "medgemma_4b",
  "dataset": "vqa_rad",
  "split": "test",
  "n": 30,
  "accuracy": 0.6333,
  "wall_seconds": 12.0,
  "metric": "closed_set_exact_match",
  "prompt": "Answer the question about the medical image. Reply with a single word or short phrase only, no explanation."
}


In [ ]:
# Robustness — limit with 150
!python scripts/paraphrase_robustness.py --model $MODEL_KEY --dataset vqa_rad --limit 150


>> robustness: medgemma_4b on vqa_rad, 150 questions x (1+4) phrasings
100% 150/150 [06:47<00:00,  2.71s/it]
{
  "model": "medgemma_4b",
  "dataset": "vqa_rad",
  "k_paraphrases": 4,
  "n": 150,
  "flip_rate": 0.4867,
  "consistency": 0.5133,
  "majority_vote_accuracy": 0.66,
  "wall_seconds": 407.2
}


In [ ]:
!echo $MODEL_KEY

medgemma_4b


In [ ]:
GPU_HOURLY = "0.80"
# Cost run
!cd "$PROJECT" && MODEL_HF="$HF_ID" GPU_HOURLY=$GPU_HOURLY bash scripts/run_cost.sh $MODEL_KEY $GPU_HOURLY
stop()

>> medgemma_4b @ concurrency=1
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x792918ef4360>, trust_remote_code=False, seed=0, num_prompts=200, dataset_name='random-mm', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', ran

In [ ]:
# Running Quantized medgemma

# FP8 — same weights, precision is the only change
MODEL_KEY = "medgemma_4b_fp8"
serve(MODEL_KEY, HF_ID, max_len=4096, quantization="fp8")


starting medgemma_4b_fp8 (quant=fp8) ... (log -> /content/vllm.log)
server up


In [ ]:
# Accuracy limit 30
!python scripts/run_accuracy.py --model $MODEL_KEY --dataset vqa_rad --limit 30

>> medgemma_4b_fp8 on vqa_rad: 30 closed-set items
100% 30/30 [00:11<00:00,  2.73it/s]
{
  "model": "medgemma_4b_fp8",
  "dataset": "vqa_rad",
  "split": "test",
  "n": 30,
  "accuracy": 0.6667,
  "wall_seconds": 11.0,
  "metric": "closed_set_exact_match",
  "prompt": "Answer the question about the medical image. Reply with a single word or short phrase only, no explanation."
}


In [ ]:
# cost run for the Quantized medgemma
! MODEL_HF="$HF_ID" GPU_HOURLY=$GPU_HOURLY ./scripts/run_cost.sh $MODEL_KEY $GPU_HOURLY

>> medgemma_4b_fp8 @ concurrency=1
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7cf7949740e0>, trust_remote_code=False, seed=0, num_prompts=200, dataset_name='random-mm', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0',

In [ ]:
# Robustness
!python scripts/paraphrase_robustness.py --model $MODEL_KEY --dataset vqa_rad --limit 150
stop()  # free the GPU before the next model

>> robustness: medgemma_4b_fp8 on vqa_rad, 150 questions x (1+4) phrasings
100% 150/150 [04:58<00:00,  1.99s/it]
{
  "model": "medgemma_4b_fp8",
  "dataset": "vqa_rad",
  "k_paraphrases": 4,
  "n": 150,
  "flip_rate": 0.48,
  "consistency": 0.52,
  "majority_vote_accuracy": 0.7067,
  "wall_seconds": 298.5
}


## 10 · Aggregate & plot

In [45]:
!python src/aggregate_results.py


          model  family dataset  accuracy  n_acc  flip_rate  consistency  cost_per_1k_cases_usd  req_throughput
medgemma_4b_fp8 unknown vqa_rad    0.6667     30     0.4800       0.5200                 0.1038           2.141
    medgemma_4b medical vqa_rad    0.6333     30     0.4867       0.5133                 0.1172           1.895
    qwen25vl_3b general vqa_rad    0.6000     30     0.4867       0.5133                 0.1063           2.091

wrote /content/drive/MyDrive/Colab Notebooks/eval with vLLM/results/summary.csv


In [46]:
def load_rungs(raw_dir):
    rungs = []
    for jf in glob.glob(f"{raw_dir}/*.json"):
        try:
            d = json.load(open(jf))
        except Exception:
            continue
        if d.get("request_throughput") is None:
            continue
        rungs.append({
            "concurrency": d.get("max_concurrency"),
            "rps": d["request_throughput"],
            "ttft_mean": d.get("mean_ttft_ms"),
            "ttft_median": d.get("median_ttft_ms"),
            "ttft_p99": d.get("p99_ttft_ms"),
        })
    return rungs

def r(x): return round(x, 1) if isinstance(x, (int, float)) else None

rows = []
for raw_dir in sorted(glob.glob(f"{PROJECT}/results/cost/raw/*")):
    key = os.path.basename(raw_dir)
    rungs = load_rungs(raw_dir)
    if not rungs:
        print("no rungs for", key); continue
    idle = min(rungs, key=lambda x: (x["concurrency"] or 1e9))
    sat  = max(rungs, key=lambda x: x["rps"])
    rows.append({
        "model": key,
        "ttft_idle_ms":     r(idle["ttft_median"] or idle["ttft_mean"]),
        "ttft_load_ms":     r(sat["ttft_median"] or sat["ttft_mean"]),
        "ttft_p99_load_ms": r(sat["ttft_p99"]),
        "sat_concurrency":  sat["concurrency"],
    })

ttft = pd.DataFrame(rows)
print(ttft.to_string(index=False))

# merge into summary.csv (replace any old ttft columns)
summ = pd.read_csv(f"{PROJECT}/results/summary.csv")
summ = summ.drop(columns=[c for c in summ.columns if c.startswith("ttft_") or c=="sat_concurrency"], errors="ignore")
summ = summ.merge(ttft, on="model", how="left")
summ.to_csv(f"{PROJECT}/results/summary.csv", index=False)
print("\nmerged. table:")
print(summ[["model","accuracy","cost_per_1k_cases_usd","ttft_idle_ms","ttft_load_ms","ttft_p99_load_ms"]].to_string(index=False))

          model  ttft_idle_ms  ttft_load_ms  ttft_p99_load_ms  sat_concurrency
    medgemma_4b         346.0        2210.1           21297.2               64
medgemma_4b_fp8         306.0        1397.8            8637.4               32
    qwen25vl_3b         213.3        1760.5            8834.6               32

merged. table:
          model  accuracy  cost_per_1k_cases_usd  ttft_idle_ms  ttft_load_ms  ttft_p99_load_ms
medgemma_4b_fp8    0.6667                 0.1038         306.0        1397.8            8637.4
    medgemma_4b    0.6333                 0.1172         346.0        2210.1           21297.2
    qwen25vl_3b    0.6000                 0.1063         213.3        1760.5            8834.6
